# 📄 Smart Document Analyst — CNN Training on Colab
**Team:** Benmouma Salma, Gassi Oumaima
**UIR S8** — AI & Big Data 2025–2026 | Prof. Hakim Hafidi

This notebook:
1. Clones the project repo
2. Downloads a subset of the RVL-CDIP dataset
3. Trains a ResNet-18 CNN for document classification
4. Evaluates with confusion matrix + classification report
5. Pushes the trained model back to GitHub

## 1️⃣ Setup — Clone Repo & Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Clone the project repository
!git clone https://github.com/OumaimaGassi/smart-document-analyst.git
%cd smart-document-analyst
!pip install -q crewai crewai-tools google-generativeai pymupdf pytesseract fpdf2 python-dotenv tqdm seaborn

## 2️⃣ Download RVL-CDIP Dataset Subset
We use HuggingFace datasets to download a manageable subset (8 classes, ~2000 images/class).

In [ ]:
!pip install -q datasets

In [ ]:
import os
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

# RVL-CDIP class names (all 16)
ALL_CLASSES = [
    'letter', 'memo', 'email', 'file_folder', 'form', 'handwritten',
    'invoice', 'advertisement', 'budget', 'news_article', 'presentation',
    'scientific_publication', 'questionnaire', 'resume', 'scientific_report', 'specification'
]

# Our subset of 8 classes
SUBSET_CLASSES = ['letter', 'form', 'invoice', 'resume', 'scientific_publication', 'email', 'memo', 'advertisement']
SUBSET_INDICES = [ALL_CLASSES.index(c) for c in SUBSET_CLASSES]
IMAGES_PER_CLASS = 2000  # Adjust if you want more/less

print(f'Target classes: {SUBSET_CLASSES}')
print(f'Images per class: {IMAGES_PER_CLASS}')

In [ ]:
# Download the dataset (streams, so it won't download everything at once)
print('Loading RVL-CDIP dataset (this may take a few minutes)...')
dataset = load_dataset('rvl_cdip', split='train', streaming=True)

# Create directories
base_dir = 'data/dataset'
for cls_name in SUBSET_CLASSES:
    os.makedirs(f'{base_dir}/{cls_name}', exist_ok=True)

# Download subset
class_counts = {c: 0 for c in SUBSET_CLASSES}
total_saved = 0
target_total = IMAGES_PER_CLASS * len(SUBSET_CLASSES)

print(f'Downloading {target_total} images...')
for sample in tqdm(dataset, total=target_total * 3, desc='Scanning'):
    label_idx = sample['label']
    if label_idx not in SUBSET_INDICES:
        continue
    
    cls_name = ALL_CLASSES[label_idx]
    if class_counts[cls_name] >= IMAGES_PER_CLASS:
        continue
    
    # Save image
    img = sample['image']
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    save_path = f'{base_dir}/{cls_name}/{cls_name}_{class_counts[cls_name]:04d}.png'
    img.save(save_path)
    class_counts[cls_name] += 1
    total_saved += 1
    
    # Check if we have enough
    if all(c >= IMAGES_PER_CLASS for c in class_counts.values()):
        break

print(f'\n✅ Downloaded {total_saved} images')
for cls, count in class_counts.items():
    print(f'   {cls}: {count}')

## 3️⃣ Train the CNN Model

In [ ]:
import sys, time, random, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm
from datetime import datetime

sys.path.insert(0, '.')
from src.models.document_classifier import DocumentClassifierCNN
from src.utils.preprocessing import DOCUMENT_CLASSES_SUBSET, CNN_INPUT_SIZE

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Data transforms
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load dataset
full_dataset = datasets.ImageFolder(root='data/dataset', transform=train_tf)
print(f'Total images: {len(full_dataset)}')
print(f'Classes: {full_dataset.classes}')

# Split: 80% train, 10% val, 10% test
n = len(full_dataset)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
n_test = n - n_train - n_val
train_set, val_set, test_set = random_split(full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED))

print(f'Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}')

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Initialize model
NUM_CLASSES = len(DOCUMENT_CLASSES_SUBSET)
model = DocumentClassifierCNN(num_classes=NUM_CLASSES, pretrained=True, dropout_rate=0.3)
model = model.to(device)

print(f'Total params: {model.get_total_params():,}')
print(f'Trainable:    {model.get_trainable_params():,}')

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

In [ ]:
# Training loop
NUM_EPOCHS = 20
PATIENCE = 5
best_val_acc = 0.0
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        _, pred = outputs.max(1)
        train_total += labels.size(0)
        train_correct += pred.eq(labels).sum().item()
    
    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            val_total += labels.size(0)
            val_correct += pred.eq(labels).sum().item()
    
    t_loss = train_loss/train_total
    t_acc = train_correct/train_total
    v_loss = val_loss/val_total
    v_acc = val_correct/val_total
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    scheduler.step()
    
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}')
    
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        patience_counter = 0
        model.save_model('model/document_classifier.pt')
        print(f'  💾 Best model saved (val_acc: {v_acc:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'\n✅ Training complete! Best val accuracy: {best_val_acc:.4f}')

## 4️⃣ Evaluation — Metrics & Confusion Matrix

In [ ]:
# Load best model and evaluate on test set
best_model = DocumentClassifierCNN.load_model('model/document_classifier.pt', NUM_CLASSES, str(device))

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc='Testing'):
        inputs = inputs.to(device)
        outputs = best_model(inputs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print('\n' + '='*60)
print('📊 CLASSIFICATION REPORT')
print('='*60)
print(classification_report(all_labels, all_preds, target_names=DOCUMENT_CLASSES_SUBSET))
print(f'Overall Accuracy: {accuracy_score(all_labels, all_preds):.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=DOCUMENT_CLASSES_SUBSET, yticklabels=DOCUMENT_CLASSES_SUBSET)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — Document Classifier\nBenmouma Salma & Gassi Oumaima')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss'])+1)
ax1.plot(epochs, history['train_loss'], 'b-o', label='Train', ms=3)
ax1.plot(epochs, history['val_loss'], 'r-o', label='Val', ms=3)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs, history['train_acc'], 'b-o', label='Train', ms=3)
ax2.plot(epochs, history['val_acc'], 'r-o', label='Val', ms=3)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Training Curves — Benmouma Salma & Gassi Oumaima')
plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Save training metadata
os.makedirs('outputs', exist_ok=True)
meta = {
    'timestamp': datetime.now().isoformat(),
    'team': 'Benmouma Salma, Gassi Oumaima',
    'accuracy': float(accuracy_score(all_labels, all_preds)),
    'best_val_acc': float(best_val_acc),
    'num_classes': NUM_CLASSES,
    'classes': DOCUMENT_CLASSES_SUBSET,
    'epochs_trained': len(history['train_loss']),
    'device': str(device),
    'pytorch_version': torch.__version__
}
with open('model/training_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('✅ Metadata saved to model/training_metadata.json')

## 5️⃣ Push Trained Model to GitHub

In [ ]:
# Configure git (change email if needed)
!git config user.name "OumaimaGassi"
!git config user.email "oumaima.gassi@uir.ac.ma"

In [ ]:
# Add and push the trained model + outputs
!git add model/document_classifier.pt model/training_metadata.json outputs/ notebooks/train_on_colab.ipynb
!git commit -m "feat: add trained CNN model (ResNet-18 on RVL-CDIP subset)" -m "Accuracy: see model/training_metadata.json"
!git push origin main

## ✅ Done!

The trained model is now on GitHub. On any computer:
```bash
git clone https://github.com/OumaimaGassi/smart-document-analyst.git
cd smart-document-analyst
pip install -r requirements.txt
cp .env.example .env  # add your GEMINI_API_KEY
python src/main.py --input path/to/document.pdf --hitl
```